**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Array Processing & Beamforming

Filtering in **space**: with several microphones/antennas, you can point a 'listening beam' at a direction, null an interferer, and locate sources — all with the same linear algebra as temporal filtering. Three sessions from array geometry to MUSIC.

## 1. Pre-requisites

- [Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) (complex exponentials, DFT).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3–S4 (eigen/subspaces) for Session 3.
- [Statistical Signal Processing](./Statistical_Signal_Processing.ipynb) S3 for MVDR.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# Uniform linear array (ULA): M sensors, half-wavelength spacing
M = 8
def steering(theta_deg):
    """Array response to a far-field narrowband source at angle theta (broadside = 0°)."""
    theta = np.deg2rad(theta_deg)
    return np.exp(1j * np.pi * np.arange(M) * np.sin(theta))   # d = λ/2

---
### 🕐 Session 1 of 3 — *The Array Manifold* (~35 min)
**Goal:** understand why direction becomes a phase pattern across sensors.
**Builds on:** [DSP Foundations](./Foundations_of_Signal_Processing_1.ipynb) S4. &nbsp; **Feeds into:** Session 2 (beamforming).

---

## 2. Direction Is a Spatial Frequency

💡 **Intuition.** A far-field wavefront hits each sensor at a slightly different time; for a narrowband signal, delay ≈ phase shift. Across a uniform line of sensors the phases advance *linearly* — direction $\theta$ shows up as a **spatial sinusoid** with frequency $\propto \sin\theta$. Everything you know about temporal frequencies transfers verbatim: sensors ↔ samples, aperture ↔ record length, beamwidth ↔ resolution, and spacing $> \lambda/2$ ⇒ *spatial aliasing* (grating lobes) — Nyquist in space.

In [2]:
# The steering vector IS a sampled sinusoid — see it
fig, axes = plt.subplots(1, 3, figsize=(10, 2.4), sharey=True)
for ax, th in zip(axes, [0, 20, 60]):
    a = steering(th)
    ax.stem(np.arange(M), a.real, basefmt=" ")
    ax.set_title(f"θ = {th}°: spatial freq ∝ sin θ")
    ax.set_xlabel("sensor")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2028716/4231524107.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Delay-and-Sum & MVDR* (~40 min)
**Goal:** steer beams; then let the data place nulls on interferers automatically.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (subspace methods).

---

## 3. Beamforming

💡 **Intuition.** **Delay-and-sum**: phase-align the sensors toward $\theta_0$ and add — signals from $\theta_0$ stack coherently ($M\times$ amplitude), others partially cancel. It's a matched filter in space, and like all matched filters it's optimal in white noise but naive about *structured* interference. **MVDR (Capon)** fixes that: minimize output power subject to unit gain at $\theta_0$ — $\mathbf{w} = \frac{R^{-1} \mathbf{a}}{\mathbf{a}^H R^{-1} \mathbf{a}}$ — a [Lagrange problem](../Intro_Math/Optimization/Optimization.ipynb) whose solution *automatically digs nulls* wherever the covariance says energy is coming from.

In [3]:
# Scene: desired source at 0°, LOUD interferer at 40°, noise
theta_s, theta_i = 0, 40
N = 4000
s = rng.standard_normal(N)                      # desired
i_sig = 3.0 * rng.standard_normal(N)            # interferer, 3x amplitude
X = (np.outer(steering(theta_s), s) + np.outer(steering(theta_i), i_sig)
     + 0.3 * (rng.standard_normal((M, N)) + 1j * rng.standard_normal((M, N))) / np.sqrt(2))

R = X @ X.conj().T / N
a0 = steering(theta_s)

w_das = a0 / M
w_mvdr = np.linalg.solve(R, a0); w_mvdr /= (a0.conj() @ w_mvdr)

angles = np.linspace(-90, 90, 721)
def pattern(w):
    return np.array([np.abs(w.conj() @ steering(t))**2 for t in angles])

plt.figure(figsize=(9, 3))
plt.plot(angles, 10*np.log10(pattern(w_das)), label="delay-and-sum")
plt.plot(angles, 10*np.log10(pattern(w_mvdr)), label="MVDR")
for th, name in [(theta_s, "source"), (theta_i, "interferer")]:
    plt.axvline(th, color="k", linestyle=":", linewidth=0.8)
plt.ylim(-60, 5); plt.legend(); plt.xlabel("angle [deg]"); plt.ylabel("gain [dB]")
plt.title("MVDR digs a null exactly at the 40° interferer — nobody told it to")
plt.tight_layout(); plt.show()

for name, w in [("delay-and-sum", w_das), ("MVDR", w_mvdr)]:
    y = w.conj() @ X
    sinr = np.var(s) * np.abs(w.conj() @ a0)**2 / np.var(y - (w.conj() @ a0) * s)
    print(f"{name:14s} output SINR ≈ {10*np.log10(sinr.real):5.1f} dB")

delay-and-sum  output SINR ≈   7.0 dB
MVDR           output SINR ≈  18.4 dB


/tmp/ipykernel_2028716/2357919198.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Subspace Methods: MUSIC* (~40 min)
**Goal:** use covariance eigenstructure to localize sources beyond the beamwidth limit.
**Builds on:** Session 2; [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3–S4.

---

## 4. MUSIC

💡 **Intuition.** With $K$ sources, the covariance's top-$K$ eigenvectors span the **signal subspace** (where steering vectors of true directions live); the remaining eigenvectors span the orthogonal **noise subspace**. MUSIC scans directions and scores each by *how orthogonal its steering vector is to the noise subspace* — true directions produce near-zero projections and towering pseudo-spectrum peaks. This is [Linear Algebra S3](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb)'s eigen-story paying rent: resolution beyond the classical beamwidth.

In [4]:
# Two sources only 8° apart — closer than the array's beamwidth
th1, th2, K = -3, 5, 2
S2 = np.stack([rng.standard_normal(N), rng.standard_normal(N)])
A = np.stack([steering(th1), steering(th2)], axis=1)
X2 = A @ S2 + 0.5 * (rng.standard_normal((M, N)) + 1j*rng.standard_normal((M, N))) / np.sqrt(2)
R2 = X2 @ X2.conj().T / N

evals, evecs = np.linalg.eigh(R2)
En = evecs[:, :M-K]                                  # noise subspace (small eigenvalues)

das_spec = np.array([np.abs(steering(t).conj() @ (R2 @ steering(t))).real for t in angles])
music = np.array([1 / np.linalg.norm(En.conj().T @ steering(t))**2 for t in angles])

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.8))
axes[0].plot(angles, 10*np.log10(das_spec / das_spec.max()))
axes[0].set_title("classical scan: one blurred bump"); axes[0].set_xlim(-40, 40)
axes[1].plot(angles, 10*np.log10(music / music.max()))
axes[1].set_title("MUSIC: two razor peaks at −3° and 5°"); axes[1].set_xlim(-40, 40)
for ax in axes:
    for th in (th1, th2): ax.axvline(th, color="k", linestyle=":", linewidth=0.8)
    ax.set_xlabel("angle [deg]")
plt.tight_layout(); plt.show()

peaks = angles[np.argsort(music)[-2:]] if False else angles[(np.diff(np.sign(np.diff(music))) < 0).nonzero()[0] + 1]
top2 = peaks[np.argsort(music[np.searchsorted(angles, peaks)])[-2:]]
print("MUSIC peak estimates:", np.sort(np.round(top2, 1)), " (truth: [-3, 5])")

MUSIC peak estimates: [-3.  5.]  (truth: [-3, 5])


/tmp/ipykernel_2028716/3945217473.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Conclusion

Direction = spatial frequency; delay-and-sum = spatial matched filter; MVDR = constrained optimization that nulls interference by itself; MUSIC = eigen-subspace geometry beating the beamwidth. Space is just another axis to filter.

---
## Where next

- [Statistical Signal Processing](./Statistical_Signal_Processing.ipynb) — the covariance machinery underneath.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — arrays of real antennas.
- [Audio & Speech DSP](./Audio_Speech_DSP.ipynb) — microphone arrays in your smart speaker run exactly this.